# 10 Document-level BioBART fine-tuning

This notebook fine-tunes BioBART for document-level biomedical text simplification using the document-level Cochrane-auto data.

It uses the full complex document as input and the simplified document as the target.

## 1. Setup and imports

In [ ]:
from __future__ import annotations

import gc
import os
import random
import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import sacrebleu
import torch
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    import evaluate
except ImportError:
    evaluate = None

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())

## 2. Load document data and sanity checks

## 2. Load document data and sanity checks

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))
print("Missing values:")
print(train_df.isna().sum().to_string())
print(val_df.isna().sum().to_string())
print(test_df.isna().sum().to_string())

In [ ]:
print("Length statistics for the complex documents")


def word_count(text: str) -> int:
    return len(str(text).split())

for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    lengths = df["complex"].fillna("").astype(str).map(word_count)
    print(split_name)
    print(lengths.describe().to_string())

print("\nSample complex document:")
print(test_df.loc[0, "complex"][:1500])
print("\nSample reference simplification:")
print(test_df.loc[0, "simple"][:1500])

## 3. Tokenization and preprocessing

## Prompt template

The model receives a full-document prompt and a target simplification.

## 4. Fine-tuning

## 3. Tokenization and preprocessing

## 5. Generation and evaluation

## 4. Fine-tuning

## 5. Generation and evaluation

In [ ]:
PROMPT_TEMPLATE = """Simplify the following biomedical document for a general audience.

Document:
{complex_document}

Simplified document:
"""


def build_prompt(complex_document: str) -> str:
    return PROMPT_TEMPLATE.format(complex_document=str(complex_document).strip())

print("Built prompt template for document-level simplification.")

In [ ]:
# Keep the training style aligned with the sentence-level BioBART notebook.
max_source_length = 512
max_target_length = 512

for candidate in MODEL_CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(candidate)
        print("Loaded tokenizer:", candidate)
        break
    except Exception as exc:
        print(f"Could not load tokenizer {candidate}: {exc}")
else:
    raise RuntimeError("Could not load any tokenizer candidate")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CANDIDATES[0])
model.to(device)

print("Loaded model:", MODEL_CANDIDATES[0])

# Basic truncation diagnostics.
def token_lengths(texts: list[str], batch_size: int = 256) -> list[int]:
    lengths = []
    for start in range(0, len(texts), batch_size):
        encoded = tokenizer(
            texts[start : start + batch_size],
            truncation=False,
            padding=False,
            add_special_tokens=True,
        )["input_ids"]
        lengths.extend(len(token_ids) for token_ids in encoded)
    return lengths

all_texts = [build_prompt(text) for text in train_df["complex"].fillna("").astype(str).tolist()]
lengths = token_lengths(all_texts)
truncated = sum(length > max_source_length for length in lengths)
print("Train prompt truncation count at 512:", truncated, "of", len(lengths))
print("Truncation rate:", round(100 * truncated / max(len(lengths), 1), 2), "%")

if truncated / max(len(lengths), 1) > 0.3:
    max_source_length = 768
    max_target_length = 768
    print("High truncation rate detected; increasing max_source_length and max_target_length to 768.")

In [ ]:

def preprocess_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    inputs = [build_prompt(text) for text in examples["complex"]]
    targets = [str(text).strip() for text in examples["simple"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding=False,
    )["input_ids"]

    labels = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label]
        for label in labels
    ]
    model_inputs["labels"] = labels
    return model_inputs


train_dataset = Dataset.from_pandas(train_df[["complex", "simple"]].copy())
val_dataset = Dataset.from_pandas(val_df[["complex", "simple"]].copy())

train_tokenized = train_dataset.map(preprocess_examples, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(preprocess_examples, batched=True, remove_columns=val_dataset.column_names)

print("Tokenized train set size:", len(train_tokenized))
print("Tokenized validation set size:", len(val_tokenized))

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=5,
    learning_rate=3e-5,
    weight_decay=0.01,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=fp16_enabled,
    bf16=bf16_enabled,
    seed=SEED,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
trainer.save_model(str(BEST_MODEL_DIR))
print("Saved best model to:", BEST_MODEL_DIR.relative_to(PROJECT_ROOT))

best_model = AutoModelForSeq2SeqLM.from_pretrained(str(BEST_MODEL_DIR)).to(device)
best_model.eval()

In [ ]:

def generate_predictions(df: pd.DataFrame) -> pd.DataFrame:
    outputs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating"):
        prompt = build_prompt(str(row["complex"]))
        inputs = tokenizer(prompt, return_tensors="pt", max_length=max_source_length, truncation=True).to(device)
        with torch.no_grad():
            generated = best_model.generate(
                **inputs,
                max_new_tokens=512,
                num_beams=4,
                length_penalty=0.9,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )
        prediction = tokenizer.decode(generated[0], skip_special_tokens=True)
        outputs.append({
            "pair_id": row["pair_id"],
            "complex": row["complex"],
            "simple": row["simple"],
            "prediction": prediction,
        })

    prediction_df = pd.DataFrame(outputs)
    prediction_df.to_csv(PREDICTION_PATH, index=False)
    print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
    return prediction_df


prediction_df = generate_predictions(test_df[["pair_id", "complex", "simple"]].copy())
print(prediction_df.head(3).to_string(index=False))

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

metrics_summary = compute_metrics(prediction_df)
print(metrics_summary)

prediction_df["source_length"] = prediction_df["complex"].fillna("").astype(str).map(word_count)
prediction_df["reference_length"] = prediction_df["simple"].fillna("").astype(str).map(word_count)
prediction_df["prediction_length"] = prediction_df["prediction"].fillna("").astype(str).map(word_count)

avg_source_length = prediction_df["source_length"].mean()
avg_reference_length = prediction_df["reference_length"].mean()
avg_prediction_length = prediction_df["prediction_length"].mean()
compression_ratio = avg_prediction_length / avg_source_length if avg_source_length else float("nan")
empty_prediction_count = int((prediction_df["prediction"].fillna("").astype(str).str.strip() == "").sum())

print("Average source length:", round(avg_source_length, 2))
print("Average reference length:", round(avg_reference_length, 2))
print("Average prediction length:", round(avg_prediction_length, 2))
print("Compression ratio:", round(compression_ratio, 3))
print("Empty prediction count:", empty_prediction_count)

In [ ]:
llama_path = RESULTS_DIR / "llama_document_level_predictions.csv"
if llama_path.exists():
    llama_df = pd.read_csv(llama_path)
    llama_df = llama_df[["pair_id", "prediction"]].rename(columns={"prediction": "llama_prediction"})
    comparison_df = prediction_df.merge(llama_df, on="pair_id", how="left")
    print(comparison_df[["pair_id", "prediction", "llama_prediction"]].head(5).to_string(index=False))
else:
    print("Llama predictions not found yet. Run notebook 09 first.")